# Phase 7 — TOR3-MALAM: controlled night preprocessing screen

**REQUIREMENT:** compare raw video frames with explicitly configured enhancement,
using the same input and detector settings. The raw baseline has no added
photometric enhancement; normal detector preprocessing still applies.

This notebook implements the **development-screen stage** only:

- `photometric_only`: native raw/transformed frame evidence on CPU.
- `detection_screen`: the same development frames plus pinned YOLO26M on GPU 0.

It does not run hold-out, tracking, ReID, geometry, crossing or counting. The next
temporal experiment requires reviewed night geometry, an explicitly selected
candidate, continuous segments and a frozen downstream profile. ReID remains in
the architecture; its final vehicle-domain configuration is OPEN.

**PROPOSAL — experimental controls:** gamma 0.7, exposure factor 1.25, contrast
factor 1.25 about 128, and unsharp masking sigma 1.0/amount 0.5, tested separately.
These are not suitable production defaults or claimed improvements.

Run **Restart Kernel → Run All**. Source outputs are intentionally empty.


## Explicit paths and execution mode

Use a complete repository checkout for this source notebook. The separately
packaged Kaggle upload contains a hash-bound source snapshot and supplies
`REPO_ROOT`; that bootstrap does not change the scientific settings.

Change `VIDEO_PATH` only with a matching sampling plan and source-video identity.
`photometric_only` needs OpenCV, NumPy and Pillow; it does not install a detector.
Choose `detection_screen` explicitly for fresh YOLO inference on the declared
15 development frames. T4x2 availability does not enable multi-GPU orchestration.


In [ ]:
from pathlib import Path

# A standalone upload bootstrap may already supply this verified source root.
if 'REPO_ROOT' not in globals():
    REPO_ROOT = Path('/kaggle/working/traffic-vehicle-counter-IDM-LOC') if Path('/kaggle/working').is_dir() else Path.cwd()
REPO_ROOT = Path(REPO_ROOT).resolve()

VIDEO_PATH = Path('/kaggle/input/datasets/chrisbiran/traffic-tracker-videos/TOR3-MALAM.mp4')
CONFIG_PATH = REPO_ROOT / 'docs/evidence/phase_7/config/TOR3-MALAM_screen_v1.json'
SAMPLING_PLAN_PATH = REPO_ROOT / 'docs/evidence/phase_7/sampling/TOR3-MALAM_sampling_plan_v1.json'
OUTPUT_ROOT = Path('/kaggle/working/phase_7_night') if Path('/kaggle/working').is_dir() else Path('/tmp/phase_7_night')
MODE = 'photometric_only'  # 'photometric_only' | 'detection_screen'
INSTALL_PINNED_DETECTOR_IF_NEEDED = True  # only used in detection_screen


## Environment and guarded source import

The notebook records actual runtime/package/GPU visibility. It does not interpret
an unavailable GPU probe as proof that GPU hardware is absent. Detection mode
separately requires PyTorch CUDA availability and the explicitly selected device.
Source tests must pass before actual-video decoding or model inference.

Hold-out is reserved from enhancement/inference and candidate selection in this
screen. Earlier raw Phase 1/geometry inspection means it is **not a blind test**.


In [ ]:
import sys
import os
import json
import gzip
import hashlib
import importlib
import importlib.metadata
import platform
import subprocess
import shutil
import time
from datetime import datetime, timezone
from copy import deepcopy

if MODE not in ('photometric_only', 'detection_screen'):
    raise ValueError('Unsupported mode; temporal/hold-out execution is not implemented here.')

HELPER_DIR = REPO_ROOT / 'notebooks'
required_source = {
    'preprocessing_helper': HELPER_DIR / 'phase_7_night_helpers.py',
    'preprocessing_tests': HELPER_DIR / 'phase_7_night_tests.py',
    'repository_notebook_source': HELPER_DIR / '07_night_preprocessing.ipynb',
}
missing = {key: str(path) for key, path in required_source.items() if not path.is_file()}
if missing:
    raise FileNotFoundError('Verified source checkout is incomplete: ' + json.dumps(missing, indent=2))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + '\n', encoding='utf-8')

source_hashes_before_guard = {key: sha256_file(path) for key, path in required_source.items()}
sys.path.insert(0, str(HELPER_DIR))
importlib.invalidate_caches()
import phase_7_night_helpers as night_helpers
import phase_7_night_tests as night_tests
night_helpers = importlib.reload(night_helpers)
night_tests = importlib.reload(night_tests)
for module in (night_helpers, night_tests):
    if Path(module.__file__).resolve().parent != HELPER_DIR:
        raise RuntimeError('Imported source belongs to another checkout; restart and Run All.')

runtime = {
    'python': sys.version, 'python_executable': sys.executable,
    'platform': platform.platform(), 'cpu_count': os.cpu_count(),
    'mode': MODE, 'package_versions': {},
    'tracking_reid_crossing_counting_executed': False,
    'holdout_evaluation_executed': False,
}
for package in ('numpy', 'Pillow', 'opencv-python', 'opencv-python-headless', 'torch', 'torchvision', 'ultralytics'):
    try:
        runtime['package_versions'][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        runtime['package_versions'][package] = None
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is None:
    runtime['gpu_probe'] = {'status': 'unknown_nvidia_smi_unavailable', 'data': None}
else:
    try:
        probe = subprocess.run([nvidia_smi, '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True, timeout=20, check=False)
        runtime['gpu_probe'] = {'status': 'reported' if probe.returncode == 0 else 'probe_failed', 'returncode': probe.returncode, 'stdout': probe.stdout, 'stderr': probe.stderr}
    except subprocess.TimeoutExpired:
        runtime['gpu_probe'] = {'status': 'probe_timeout', 'data': None}
print(json.dumps(runtime, indent=2))


In [ ]:
guard_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
GUARD_DIR = OUTPUT_ROOT / ('guard_' + guard_id)
GUARD_DIR.mkdir(parents=True, exist_ok=False)
write_json(GUARD_DIR / 'runtime_before_detector_setup.json', runtime)
write_json(GUARD_DIR / 'source_code_sha256.json', source_hashes_before_guard)
test_report = night_tests.run_tests()
write_json(GUARD_DIR / 'synthetic_test_results.json', test_report)
print(json.dumps({key: test_report.get(key) for key in ('successful', 'tests_run', 'tests_passed', 'failures', 'errors')}, indent=2))
if not test_report['successful']:
    raise AssertionError('Software guard failed; actual-video work is blocked: ' + str(GUARD_DIR))
if {key: sha256_file(path) for key, path in required_source.items()} != source_hashes_before_guard:
    raise RuntimeError('Source changed while tests ran; restart and rerun the guard.')


## Frozen experimental configuration and sampling isolation

The plan declares both future continuous segments and development-screen frame
indices. Only the development indices are retained, enhanced or inferred here. Sequential decoder prefix reads can pass through reserved frames without retaining or evaluating them. Hold-out frame
indices are recorded as reserved, and enhancement selection remains OPEN.

Downstream counting profile: **B v1 + explicit review overlay, offline**, with A
retained for comparison. Its numeric association controls remain frozen. That
profile is not executed or transferred to the night video without review.


In [ ]:
for name, path in (('configuration', CONFIG_PATH), ('sampling plan', SAMPLING_PLAN_PATH)):
    if not path.is_file():
        raise FileNotFoundError(name + ' is missing: ' + str(path))
config_bytes = CONFIG_PATH.read_bytes()
sampling_bytes = SAMPLING_PLAN_PATH.read_bytes()
config = json.loads(config_bytes)
sampling_plan = json.loads(sampling_bytes)
if sha256_file(SAMPLING_PLAN_PATH) != config['expected_sampling_sha256']:
    raise ValueError('Sampling-plan bytes differ from the frozen configuration.')
if config['expected_video_sha256'] != 'a92e693b208813cccbac3921a5169b08443fdf425e352168d0ae9db3fe4f2082':
    raise ValueError('This prepared screen is bound to the retained TOR3-MALAM source.')
if config['variants'] != night_helpers.DEFAULT_VARIANTS:
    raise ValueError('Candidate transformations differ from the documented v1 screen.')
if config['detector']['settings'] != night_helpers.DEFAULT_DETECTOR_SETTINGS:
    raise ValueError('Detector settings differ from the frozen Phase 2 control.')
if config['detector']['package_version'] != '8.4.175' or config['detector']['device'] != 0:
    raise ValueError('Detection screen requires pinned Ultralytics 8.4.175 and explicit device 0.')
dev_indices = night_helpers.verify_sampling_plan(sampling_plan)
print('Development frame indices:', dev_indices)
print('Hold-out is reserved; no hold-out retention, enhancement or inference is requested.')
(GUARD_DIR / 'configuration_input.json').write_bytes(config_bytes)
(GUARD_DIR / 'sampling_plan_input.json').write_bytes(sampling_bytes)


## Detector setup — detection mode only

This stage verifies official weight bytes before loading. It keeps the established
Phase 2 full-frame settings and raw model classes. `quantize=None` denotes the
native FP32 control in the pinned package; it does not use deprecated `half`.

When needed, the notebook installs only `ultralytics==8.4.175` with `--no-deps`.
Existing Kaggle Torch/OpenCV/NumPy packages are not silently upgraded. A missing
runtime dependency fails explicitly rather than changing the experiment.


In [ ]:
package_setup = {'detector_package_installation_executed': False}
if MODE == 'detection_screen':
    expected_version = config['detector']['package_version']
    try:
        installed_version = importlib.metadata.version('ultralytics')
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        if not INSTALL_PINNED_DETECTOR_IF_NEEDED:
            raise RuntimeError('Install ultralytics==' + expected_version + ' before this mode.')
        # Replace only this explicit package, preserving installed Kaggle dependencies.
        install = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'ultralytics==' + expected_version], capture_output=True, text=True, check=False)
        package_setup = {'detector_package_installation_executed': True, 'returncode': install.returncode, 'stdout': install.stdout, 'stderr': install.stderr}
        write_json(GUARD_DIR / 'detector_package_installation.json', package_setup)
        if install.returncode != 0:
            raise RuntimeError('Pinned detector installation failed; see guard artifacts.')
    importlib.invalidate_caches()
    if importlib.metadata.version('ultralytics') != expected_version:
        raise RuntimeError('Pinned package verification failed.')
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() <= config['detector']['device']:
        raise RuntimeError('Detection mode requires visible PyTorch CUDA device 0; no automatic CPU fallback.')
print(json.dumps(package_setup, indent=2))
# The helper owns official weights verification, actual GPU/model creation,
# effective predictor checks, warm-up and measured resource evidence.


## Execute the development screen

Every variant uses the same source frame indices and native dimensions. Raw and
transformed images are retained separately with byte hashes. Detection boxes are
frame-level observations; their number is not a vehicle count or a model-quality
score. The helper records measured processing time/resource data and limitations.


In [ ]:
if {key: sha256_file(path) for key, path in required_source.items()} != source_hashes_before_guard:
    raise RuntimeError('Tested source changed before execution.')
if CONFIG_PATH.read_bytes() != config_bytes or SAMPLING_PLAN_PATH.read_bytes() != sampling_bytes:
    raise RuntimeError('Frozen inputs changed before execution.')
helper_mode = {'photometric_only': 'photometric', 'detection_screen': 'detection'}[MODE]
execution_started_utc = datetime.now(timezone.utc).isoformat()
clock = time.perf_counter()
report = night_helpers.run_screen(VIDEO_PATH, SAMPLING_PLAN_PATH, OUTPUT_ROOT, config=config, mode=helper_mode)
measured_screen_wall_seconds = time.perf_counter() - clock
RUN_DIR = Path(report['output_directory'])
if not RUN_DIR.is_dir():
    raise RuntimeError('Helper did not retain the declared run directory.')
print('Actual run directory:', RUN_DIR)
print(json.dumps(report, indent=2))


In [ ]:
source_snapshot_dir = RUN_DIR / 'source_snapshot'
source_snapshot_dir.mkdir(exist_ok=False)
for key, path in required_source.items():
    if sha256_file(path) != source_hashes_before_guard[key]:
        raise RuntimeError('Source changed after execution; preserve evidence and investigate.')
    with (source_snapshot_dir / (path.name + '.gz')).open('wb') as stream:
        with gzip.GzipFile(filename='', mode='wb', fileobj=stream, mtime=0) as compressed:
            compressed.write(path.read_bytes())
shutil.copytree(GUARD_DIR, RUN_DIR / 'pre_execution_guard')
notebook_manifest = {
    'claim_type': 'ACTUAL_EXECUTION_DEVELOPMENT_SCREEN_NOT_MODEL_BENCHMARK',
    'execution_started_utc': execution_started_utc,
    'execution_finished_utc': datetime.now(timezone.utc).isoformat(),
    'measured_screen_wall_seconds': measured_screen_wall_seconds,
    'mode': MODE, 'runtime': runtime, 'package_setup': package_setup,
    'actual_helper_model_resource': report['resource'],
    'source_code_sha256': source_hashes_before_guard,
    'configuration_sha256': hashlib.sha256(config_bytes).hexdigest(),
    'sampling_plan_sha256': hashlib.sha256(sampling_bytes).hexdigest(),
    'test_guard': {'successful': test_report['successful'], 'tests_run': test_report['tests_run']},
    'tracking_reid_geometry_crossing_counting_executed': False,
    'holdout_evaluation_executed': False, 'selected_enhancement': None,
    'next_stage_gate': 'NIGHT_GEOMETRY_REVIEW_AND_CANDIDATE_SELECTION_REQUIRED',
    'decoder_prefix_read_through_holdout_is_not_evaluation': True,
    'formal_ground_truth_available': False,
    'image_quality_statistics_are_not_detection_or_counting_accuracy': True,
}
write_json(RUN_DIR / 'notebook_execution_manifest.json', notebook_manifest)
print('Retained source, guard and runtime:', RUN_DIR)


## Review the retained evidence

Compare **the same frame** before and after enhancement. Look for recoverable
vehicle detail and regressions such as amplified glare/noise, clipped highlights,
extra boxes or category changes. A brighter image, higher confidence or more
boxes alone does not establish benefit. Development-frame outputs cannot establish
tracking, ReID, physical-passage identity or counting behavior.


In [ ]:
from PIL import Image as PILImage, ImageDraw
from IPython.display import Image, display

# Display-only overviews; native source/transformed/annotated PNGs remain separate.
# Representative rows are selected by position in the frozen DEV list, not scores.
row_positions = sorted(set([0, min(2, len(dev_indices)-1), len(dev_indices)//2, len(dev_indices)-3]))
preview_indices = [dev_indices[position] for position in row_positions]
variant_entries = report['variants']
thumbnail_width, thumbnail_height = 256, 144
header_height, row_label_height = 38, 27
for image_kind in ('transformed', 'annotated'):
    if image_kind == 'annotated' and MODE != 'detection_screen':
        continue
    overview = PILImage.new('RGB', (thumbnail_width * len(variant_entries), header_height + (thumbnail_height + row_label_height) * len(preview_indices)), (22, 22, 22))
    drawing = ImageDraw.Draw(overview)
    for column, entry in enumerate(variant_entries):
        drawing.text((column*thumbnail_width+8, 10), entry['variant']['name'], fill='white')
        indexed = {sample['frame_index']: sample for sample in entry['samples']}
        for row, frame_index in enumerate(preview_indices):
            source_image_path = Path(indexed[frame_index][image_kind]['path'])
            if sha256_file(source_image_path) != indexed[frame_index][image_kind]['sha256']:
                raise ValueError('Retained image bytes changed before display.')
            y = header_height + row * (thumbnail_height + row_label_height)
            drawing.text((column*thumbnail_width+8, y+6), 'frame ' + str(frame_index), fill='white')
            with PILImage.open(source_image_path) as native:
                preview = native.convert('RGB').resize((thumbnail_width, thumbnail_height), PILImage.Resampling.LANCZOS)
            overview.paste(preview, (column*thumbnail_width, y+row_label_height))
    preview_path = RUN_DIR / ('development_' + image_kind + '_overview.png')
    overview.save(preview_path)
    print(preview_path.relative_to(RUN_DIR), 'display-only resized evidence')
    display(Image(filename=str(preview_path)))

artifacts = []
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file() and path.name != 'notebook_archive_manifest.json':
        artifacts.append({'path': str(path.relative_to(RUN_DIR)), 'size_bytes': path.stat().st_size, 'sha256': sha256_file(path)})
write_json(RUN_DIR / 'notebook_archive_manifest.json', {'run_dir': str(RUN_DIR), 'artifacts': artifacts})
print('Enhancement selection is OPEN. No hold-out evaluation or temporal run has been executed.')


## Next action

1. Review the separate TOR3-MALAM candidate geometry image and direction mapping.
2. Review these development images/detections and record candidate selection or
   retain raw; if evidence is insufficient, keep selection OPEN.
3. Prepare a separate controlled continuous-segment experiment with identical
   detector, tracker, ReID and counting settings across raw/selected enhancement.
4. Run untouched hold-out only after development choices are frozen.

No accuracy, precision, recall, F1, MAPE, tracking accuracy or counting accuracy is
reported. No raw video or model weights should be committed to GitHub.
